# Course 07: TensorFlow on Google Cloud

**Companion notebook for**: `07-tensorflow-gcp.html`

## Learning Objectives
- Build efficient input pipelines with `tf.data.Dataset` (map, batch, prefetch, cache, interleave)
- Construct Keras models using Sequential, Functional, and Subclassing APIs
- Train a model on Fashion MNIST with callbacks (EarlyStopping, ModelCheckpoint, TensorBoard, ReduceLROnPlateau)
- Understand distributed training strategy patterns (MirroredStrategy, MultiWorkerMirroredStrategy, TPUStrategy)
- Configure Vertex AI custom training jobs and hyperparameter tuning (SDK code, commented for cost)

## Exam Relevance
- **Section 3**: Scaling prototypes into ML models

## Prerequisites
- Python 3.10+
- TensorFlow 2.12+

---
## Setup & Dependencies

In [ ]:
%pip install -q tensorflow numpy matplotlib

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import time

print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs available: {len(tf.config.list_physical_devices('GPU'))}")
print(f"Eager execution: {tf.executing_eagerly()}")

---
## Section 1: tf.data Pipeline — Create, Map, Batch, Prefetch

The `tf.data.Dataset` API builds efficient, composable input pipelines. The key operations are:

- **`from_tensor_slices()`** — create a dataset from in-memory data
- **`map()`** — apply a transformation to each element
- **`batch()`** — combine elements into batches
- **`prefetch()`** — overlap data preparation with model training
- **`cache()`** — store elements in memory after first read
- **`interleave()`** — read from multiple sources in parallel

### Optimal Pipeline Order
```
list_files → interleave → map (parse) → cache → map (augment) → shuffle → batch → prefetch
```

In [ ]:
# --- Basic tf.data pipeline from in-memory data ---

# Create sample data
np.random.seed(42)
X = np.random.randn(1000, 10).astype(np.float32)
y = (X[:, 0] + X[:, 1] > 0).astype(np.int32)  # Binary classification

# Create dataset from tensors
dataset = tf.data.Dataset.from_tensor_slices((X, y))
print(f"Dataset element spec: {dataset.element_spec}")
print(f"Dataset cardinality: {dataset.cardinality().numpy()}")

# Inspect a single element
for features, label in dataset.take(1):
    print(f"\nSingle element — features shape: {features.shape}, label: {label.numpy()}")

In [ ]:
# --- Apply transformations: map, batch, prefetch ---

def normalize(features, label):
    """Normalize features to zero mean, unit variance."""
    mean = tf.constant([0.0] * 10)  # Precomputed in production
    std = tf.constant([1.0] * 10)
    return (features - mean) / std, label


# Build the pipeline
train_ds = (
    tf.data.Dataset.from_tensor_slices((X[:800], y[:800]))
    .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)  # Parallel map
    .cache()                          # Cache after parsing/normalization
    .shuffle(buffer_size=800)         # Shuffle after cache
    .batch(32)                        # Batch
    .prefetch(tf.data.AUTOTUNE)       # Prefetch for pipeline overlap
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X[800:], y[800:]))
    .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)

# Verify batch shapes
for batch_x, batch_y in train_ds.take(1):
    print(f"Batch features shape: {batch_x.shape}")
    print(f"Batch labels shape:   {batch_y.shape}")
    print(f"Label distribution:   {tf.reduce_sum(batch_y).numpy()} positive out of {len(batch_y)}")

In [ ]:
# --- Benchmark: prefetch vs no prefetch ---

def simulate_training_step(batch):
    """Simulate a compute-heavy training step."""
    time.sleep(0.01)  # 10ms simulated GPU work


def benchmark_pipeline(dataset, name, num_batches=50):
    """Measure end-to-end throughput of a pipeline."""
    start = time.perf_counter()
    for i, batch in enumerate(dataset):
        if i >= num_batches:
            break
        simulate_training_step(batch)
    elapsed = time.perf_counter() - start
    print(f"{name:<30} {elapsed:.3f}s  ({num_batches / elapsed:.1f} batches/sec)")
    return elapsed


# Create two versions of the same dataset
base = tf.data.Dataset.from_tensor_slices((X, y)).batch(32)

no_prefetch = base  # No prefetching
with_prefetch = base.prefetch(tf.data.AUTOTUNE)  # With prefetching

print("Pipeline Throughput Comparison:")
print("-" * 55)
t1 = benchmark_pipeline(no_prefetch, "Without prefetch")
t2 = benchmark_pipeline(with_prefetch, "With prefetch(AUTOTUNE)")
print(f"\nSpeedup: {t1/t2:.2f}x")
print("\nNote: In real training with GPU compute, prefetch overlap is more significant.")

---
## Section 2: Keras Model — Build, Compile, Train on Fashion MNIST

We will demonstrate all three Keras APIs (Sequential, Functional, Subclassing) and then train on Fashion MNIST to compare training with and without callbacks.

In [ ]:
# --- Load Fashion MNIST ---

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Normalize pixel values to [0, 1]
X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

# Add channel dimension for CNN compatibility
X_train = X_train[..., np.newaxis]  # (60000, 28, 28, 1)
X_test = X_test[..., np.newaxis]

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print(f"Training set:   {X_train.shape}, labels: {y_train.shape}")
print(f"Test set:       {X_test.shape}, labels: {y_test.shape}")
print(f"Pixel range:    [{X_train.min()}, {X_train.max()}]")
print(f"Classes:        {len(class_names)}")

# Show sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i, :, :, 0], cmap="gray")
    ax.set_title(class_names[y_train[i]], fontsize=10)
    ax.axis("off")
plt.suptitle("Fashion MNIST — Sample Images", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# --- Build tf.data pipelines for training and validation ---

BATCH_SIZE = 64
VALIDATION_SPLIT = 0.1
NUM_VAL = int(len(X_train) * VALIDATION_SPLIT)

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train[NUM_VAL:], y_train[NUM_VAL:]))
    .cache()
    .shuffle(10000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_train[:NUM_VAL], y_train[:NUM_VAL]))
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((X_test, y_test))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print(f"Training batches:   {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Validation batches: {tf.data.experimental.cardinality(val_ds).numpy()}")
print(f"Test batches:       {tf.data.experimental.cardinality(test_ds).numpy()}")

In [ ]:
# --- API 1: Sequential Model ---

sequential_model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

sequential_model.summary()
print("\nSequential: best for simple linear stacks. No branching, no shared layers.")

In [ ]:
# --- API 2: Functional Model (with residual connection) ---

inputs = tf.keras.Input(shape=(28, 28, 1))
x = tf.keras.layers.Flatten()(inputs)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)

# Residual connection (skip connection)
residual = x
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Add()([x, residual])  # Residual add
x = tf.keras.layers.LayerNormalization()(x)

outputs = tf.keras.layers.Dense(10, activation="softmax")(x)

functional_model = tf.keras.Model(inputs=inputs, outputs=outputs, name="functional_with_residual")
functional_model.summary()
print("\nFunctional: supports residual connections, multi-input/output, shared layers.")

In [ ]:
# --- API 3: Model Subclassing ---

class FashionClassifier(tf.keras.Model):
    """Custom model with dynamic architecture via subclassing."""

    def __init__(self, num_classes=10, hidden_units=128, dropout_rate=0.3):
        super().__init__()
        self.flatten = tf.keras.layers.Flatten()
        self.dense1 = tf.keras.layers.Dense(hidden_units, activation="relu")
        self.dropout = tf.keras.layers.Dropout(dropout_rate)
        self.dense2 = tf.keras.layers.Dense(hidden_units, activation="relu")
        self.norm = tf.keras.layers.LayerNormalization()
        self.classifier = tf.keras.layers.Dense(num_classes, activation="softmax")

    def call(self, inputs, training=False):
        x = self.flatten(inputs)
        x = self.dense1(x)
        x = self.dropout(x, training=training)

        # Residual connection with Python control flow
        residual = x
        x = self.dense2(x)
        x = x + residual
        x = self.norm(x)

        return self.classifier(x)


subclass_model = FashionClassifier()
# Build the model by passing sample data
subclass_model(tf.zeros((1, 28, 28, 1)))
subclass_model.summary()
print("\nSubclassing: full Python flexibility, but harder to serialize/inspect.")

In [ ]:
# --- Compare all three APIs ---

print(f"{'API':<20} {'Parameters':>12} {'Best For'}")
print("-" * 65)
print(f"{'Sequential':<20} {sequential_model.count_params():>12,} {'Simple feed-forward'}")
print(f"{'Functional':<20} {functional_model.count_params():>12,} {'Multi-input, residual, shared layers'}")
print(f"{'Subclassing':<20} {subclass_model.count_params():>12,} {'Dynamic arch, research, GANs'}")
print("\nExam tip: Functional API is the default for production. Use Sequential for")
print("prototyping and Subclassing only when you need Python control flow in forward pass.")

---
## Section 3: Training Callbacks Demo

Callbacks hook into the training loop to perform actions at epoch/batch boundaries:

| Callback | Purpose |
|---|---|
| `EarlyStopping` | Stop training when validation metric stops improving |
| `ModelCheckpoint` | Save best model weights during training |
| `TensorBoard` | Log metrics for visualization |
| `ReduceLROnPlateau` | Reduce learning rate when metric plateaus |

In [ ]:
# --- Train the Functional model WITH callbacks ---

model = tf.keras.models.clone_model(functional_model)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        min_delta=0.001,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath="best_fashion_model.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.TensorBoard(
        log_dir="./logs/fashion_mnist",
        histogram_freq=1
    ),
]

print("Training with callbacks (EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard):")
print("=" * 80)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,  # Set high; EarlyStopping will halt training
    callbacks=callbacks
)

In [ ]:
# --- Visualize training history ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history["loss"], label="Train Loss", color="#3b82f6", linewidth=2)
axes[0].plot(history.history["val_loss"], label="Val Loss", color="#ef4444", linewidth=2)
axes[0].set_title("Loss", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history.history["accuracy"], label="Train Accuracy", color="#3b82f6", linewidth=2)
axes[1].plot(history.history["val_accuracy"], label="Val Accuracy", color="#22c55e", linewidth=2)
axes[1].set_title("Accuracy", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Fashion MNIST Training with Callbacks", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

# Evaluate on test set
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"\nTest accuracy: {test_acc:.4f}")
print(f"Test loss:     {test_loss:.4f}")
print(f"\nTraining stopped at epoch {len(history.history['loss'])} (EarlyStopping patience=5)")

---
## Section 4: Distributed Strategy Comparison

TensorFlow provides distribution strategies for scaling training across multiple devices. Below we show the **code patterns** for each strategy (actual multi-GPU/TPU execution requires the corresponding hardware).

| Strategy | Hardware | Use Case |
|---|---|---|
| `MirroredStrategy` | 1 machine, N GPUs | Most common multi-GPU |
| `MultiWorkerMirroredStrategy` | M machines, N GPUs each | Scale beyond single machine |
| `TPUStrategy` | TPU pod | Large-batch training |

In [ ]:
# --- Strategy 1: MirroredStrategy (single machine, multi-GPU) ---

# This works even with 0 or 1 GPU (falls back to default device)
mirrored_strategy = tf.distribute.MirroredStrategy()
print(f"MirroredStrategy replicas: {mirrored_strategy.num_replicas_in_sync}")

# Everything inside strategy.scope() is replicated across devices
with mirrored_strategy.scope():
    mirrored_model = tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(10, activation="softmax")
    ])
    mirrored_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

# Scale batch size by number of replicas
GLOBAL_BATCH = BATCH_SIZE * mirrored_strategy.num_replicas_in_sync
print(f"Global batch size: {GLOBAL_BATCH} (per-replica: {BATCH_SIZE})")
print("\nKey: model.compile() and model.fit() work unchanged inside strategy.scope().")

In [ ]:
# --- Strategy 2: MultiWorkerMirroredStrategy (code pattern) ---
# This requires TF_CONFIG environment variable set on each worker.
# Shown here as documentation — do NOT run on a single machine.

print("MultiWorkerMirroredStrategy — Code Pattern")
print("=" * 50)

multi_worker_code = '''
import os
import json
import tensorflow as tf

# TF_CONFIG must be set on each worker (typically by orchestrator)
# Example for worker 0 of 2:
os.environ["TF_CONFIG"] = json.dumps({
    "cluster": {
        "worker": ["worker0.example.com:12345",
                   "worker1.example.com:12345"]
    },
    "task": {"type": "worker", "index": 0}
})

strategy = tf.distribute.MultiWorkerMirroredStrategy()

with strategy.scope():
    model = build_model()
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

# Each worker processes a shard of the data
# Global batch = per_worker_batch * num_workers
model.fit(train_dataset, epochs=10)
'''
print(multi_worker_code)
print("Key: TF_CONFIG tells each worker about the cluster topology.")
print("Vertex AI sets TF_CONFIG automatically when you specify replica_count > 1.")

In [ ]:
# --- Strategy 3: TPUStrategy (code pattern) ---

print("TPUStrategy — Code Pattern")
print("=" * 50)

tpu_code = '''
import tensorflow as tf

# Step 1: Resolve TPU cluster
resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
tf.config.experimental_connect_to_cluster(resolver)
tf.tpu.experimental.initialize_tpu_system(resolver)

# Step 2: Create TPU strategy
strategy = tf.distribute.TPUStrategy(resolver)
print(f"TPU cores: {strategy.num_replicas_in_sync}")  # Usually 8 per TPU

# Step 3: Build model inside strategy scope
with strategy.scope():
    model = build_model()
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

# TPU tips:
# - Use tf.float32 carefully (TPUs prefer bfloat16 for matmuls)
# - Avoid Python-side data processing in map functions
# - Batch size should be a multiple of 8 (one per TPU core)
# - Use tf.data pipelines, not feed_dict or numpy arrays
model.fit(train_dataset, epochs=10)
'''
print(tpu_code)
print("Key: TPUs use bfloat16 natively. Batch size must be divisible by num_replicas.")

In [ ]:
# --- Strategy comparison summary ---

print("Distribution Strategy Decision Tree")
print("=" * 60)
print()
print("Hardware Setup              -> Strategy")
print("-" * 60)
print("Single GPU                  -> No strategy needed")
print("1 machine, 2-8 GPUs         -> MirroredStrategy")
print("2+ machines, N GPUs each    -> MultiWorkerMirroredStrategy")
print("TPU v2/v3/v4 pod            -> TPUStrategy")
print("Parameter server setup      -> ParameterServerStrategy")
print()
print("Batch Size Scaling Rule:")
print("  global_batch = per_replica_batch * num_replicas")
print("  learning_rate = base_lr * num_replicas  (linear scaling)")
print("  Use warmup epochs when scaling to many replicas")

---
## Section 5: Vertex AI Custom Training Job (SDK Code)

The code below shows how to submit training jobs to Vertex AI using the Python SDK. These calls are **commented out** to avoid incurring cloud costs.

### Pre-built vs Custom Containers
- **Pre-built**: Google-maintained images with TF/PyTorch/XGBoost pre-installed. Use when standard frameworks suffice.
- **Custom**: Your own Docker image. Use when you have proprietary dependencies or non-standard frameworks.

In [ ]:
# --- Vertex AI Custom Training Job (commented out to avoid cost) ---

vertex_training_code = '''
from google.cloud import aiplatform

aiplatform.init(project="my-project", location="us-central1")

# --- Option 1: Pre-built container ---
job = aiplatform.CustomTrainingJob(
    display_name="fashion-mnist-tf",
    script_path="trainer/task.py",  # Your training script
    container_uri="us-docker.pkg.dev/vertex-ai/training/tf-gpu.2-12:latest",
    requirements=["cloudml-hypertune"],
    model_serving_container_image_uri=(
        "us-docker.pkg.dev/vertex-ai/prediction/tf2-gpu.2-12:latest"
    ),
)

model = job.run(
    replica_count=1,
    machine_type="n1-standard-8",
    accelerator_type="NVIDIA_TESLA_V100",
    accelerator_count=2,
    args=["--epochs=50", "--batch-size=128"],
)

# --- Option 2: Custom container ---
custom_job = aiplatform.CustomContainerTrainingJob(
    display_name="custom-fashion-mnist",
    container_uri="us-docker.pkg.dev/my-project/my-repo/trainer:latest",
    model_serving_container_image_uri=(
        "us-docker.pkg.dev/vertex-ai/prediction/tf2-gpu.2-12:latest"
    ),
)

# Multi-worker distributed training
model = custom_job.run(
    replica_count=4,  # 4 workers
    machine_type="n1-standard-8",
    accelerator_type="NVIDIA_TESLA_V100",
    accelerator_count=2,  # 2 GPUs per worker = 8 total
)
# Vertex AI automatically sets TF_CONFIG for MultiWorkerMirroredStrategy
'''

print("Vertex AI Custom Training Job — Code Template")
print("=" * 55)
print(vertex_training_code)
print("NOTE: This code is for reference only. Running it will incur cloud costs.")

---
## Section 6: Hyperparameter Tuning Config Example

Vertex AI uses the **Vizier** service for hyperparameter optimization. You define:
- **metric_spec**: which metric to optimize and direction (maximize/minimize)
- **parameter_spec**: hyperparameters, their types, ranges, and scales
- **max_trial_count**: total trials to run
- **parallel_trial_count**: trials running concurrently

In [ ]:
# --- Hyperparameter Tuning Job Config (commented out) ---

hpt_code = '''
from google.cloud import aiplatform
from google.cloud.aiplatform import hyperparameter_tuning as hpt

aiplatform.init(project="my-project", location="us-central1")

# Define the base training job
custom_job = aiplatform.CustomJob(
    display_name="hpt-base-job",
    worker_pool_specs=[{
        "machine_spec": {
            "machine_type": "n1-standard-8",
            "accelerator_type": "NVIDIA_TESLA_V100",
            "accelerator_count": 1,
        },
        "replica_count": 1,
        "container_spec": {
            "image_uri": "us-docker.pkg.dev/vertex-ai/training/tf-gpu.2-12:latest",
            "command": ["python", "trainer/task.py"],
        },
    }],
)

# Hyperparameter tuning job
hpt_job = aiplatform.HyperparameterTuningJob(
    display_name="fashion-mnist-hpt",
    custom_job=custom_job,
    metric_spec={"val_accuracy": "maximize"},
    parameter_spec={
        "learning_rate": hpt.DoubleParameterSpec(
            min=1e-5, max=1e-1, scale="log"
        ),
        "num_layers": hpt.IntegerParameterSpec(
            min=1, max=5, scale="linear"
        ),
        "hidden_units": hpt.DiscreteParameterSpec(
            values=[64, 128, 256, 512], scale="linear"
        ),
        "dropout_rate": hpt.DoubleParameterSpec(
            min=0.1, max=0.5, scale="linear"
        ),
        "optimizer": hpt.CategoricalParameterSpec(
            values=["adam", "sgd", "rmsprop"]
        ),
    },
    max_trial_count=50,
    parallel_trial_count=5,
    search_algorithm=None,  # None = Bayesian (Vizier default)
)

hpt_job.run()

# In trainer/task.py, report metrics using cloudml-hypertune:
# import hypertune
# hpt = hypertune.HyperTune()
# hpt.report_hyperparameter_tuning_metric(
#     hyperparameter_metric_tag="val_accuracy",
#     metric_value=val_accuracy,
#     global_step=epoch
# )
'''

print("Vertex AI Hyperparameter Tuning — Code Template")
print("=" * 55)
print(hpt_code)
print("NOTE: This code is for reference. Running it incurs cloud costs.")
print("\nParameter scale guide:")
print("  'log'    — for learning rate, weight decay (spans orders of magnitude)")
print("  'linear' — for dropout rate, num layers (uniform range)")
print("  Vizier's Bayesian optimizer is usually better than grid/random search.")

---
## Summary

In this notebook we covered:

1. **tf.data Pipeline** — Created datasets from tensors, applied map/batch/prefetch/cache transformations, and benchmarked prefetch performance.

2. **Keras Model APIs** — Built identical models using Sequential, Functional, and Subclassing APIs. Functional API is the production default.

3. **Training Callbacks** — Trained on Fashion MNIST with EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, and TensorBoard logging.

4. **Distributed Strategies** — Demonstrated MirroredStrategy live, showed code patterns for MultiWorkerMirroredStrategy and TPUStrategy.

5. **Vertex AI Training** — Showed SDK code for custom training jobs (pre-built and custom containers) and hyperparameter tuning with Vizier.

### Exam Key Points
- **tf.data order**: interleave > map > cache > shuffle > batch > prefetch
- **MirroredStrategy**: single machine, multi-GPU
- **MultiWorkerMirroredStrategy**: multi-machine (requires TF_CONFIG)
- **TPUStrategy**: TPU pods (prefer bfloat16)
- **Pre-built containers**: standard frameworks. **Custom containers**: proprietary dependencies.

**Next notebook**: Course 08 covers Production Machine Learning Systems.